# Grad-CAM de re-ID — ¿qué mira cada modelo para decir que dos imágenes son la misma vaca?

Como los encoders de re-ID no tienen clase de salida (dan un embedding de 2048-d), el Grad-CAM
se calcula sobre la **similitud coseno entre el par** (una imagen y otra del mismo individuo):
el gradiente de esa similitud respecto a las activaciones muestra **qué regiones hacen que el
modelo las considere la misma vaca**.

Compara **tu encoder (cmpd300_source.pt)** vs **ResNet-50 de ImageNet puro**, sobre las mismas
imágenes, en caras (Ahmed) y hocicos (Zenodo). Sirve para *explicar* por qué ImageNet empata:
si los dos miran pelaje/fondo/contexto y no el hocico, ahí está la respuesta visual.

Prerrequisitos en Drive: `dataset_caras_bovinos.zip`, `BeefCattle_Muzzle_Individualized.zip`,
`outputs/checkpoints/cmpd300_source.pt`, y el código de Fase 6 commiteado.

## 0. GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'Activá GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Drive

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
assert DRIVE_ROOT.is_dir()

## 2. Código (clon limpio del fork)

In [ ]:
import os, shutil
os.chdir('/content')
REPO_URL='https://github.com/trinidadpujol/tp-final-vision2-Pujol-Vitale.git'
REPO_DIR='/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone -b rama_trini1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
import sys; sys.path.insert(0, REPO_DIR)

## 3. Symlink outputs (trae el checkpoint de la Fase 5)

In [ ]:
for sub in ('checkpoints','logs','results'):
    d=DRIVE_ROOT/'outputs'/sub; d.mkdir(parents=True, exist_ok=True)
    l=Path(REPO_DIR)/'outputs'/sub
    if l.exists() and not l.is_symlink(): shutil.rmtree(l)
    if not l.exists(): os.symlink(d,l,target_is_directory=True)
assert (Path(REPO_DIR)/'outputs/checkpoints/cmpd300_source.pt').is_file(), 'falta el checkpoint'
print('ok')

## 4. Datos: subset de caras (Ahmed) y hocicos (Zenodo)

In [ ]:
import zipfile, random
IMG_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}

# --- Ahmed (caras): subset chico ---
AH=Path('/content/ahmed_gc')
if AH.exists(): shutil.rmtree(AH)
AH.mkdir(parents=True)
with zipfile.ZipFile(DRIVE_ROOT/'dataset_caras_bovinos.zip') as z:
    mem=[m for m in z.namelist() if not m.endswith('/') and Path(m).suffix.lower() in IMG_EXTS]
    byid={}
    for m in mem: byid.setdefault(Path(m).parent.as_posix(),[]).append(m)
    ids=list(byid); random.Random(1).shuffle(ids)
    for ind in ids[:20]:
        nm=Path(ind).name; (AH/nm).mkdir(exist_ok=True)
        for m in byid[ind][:6]: (AH/nm/Path(m).name).write_bytes(z.read(m))
print('caras:', sum(1 for _ in AH.iterdir()),'individuos')

# --- Zenodo (hocicos): extracción completa (chico) ---
ZE=Path('/content/zenodo_gc')
if not ZE.exists():
    ZE.mkdir(parents=True)
    !unzip -q "{DRIVE_ROOT}/BeefCattle_Muzzle_Individualized.zip" -d "{ZE}"
cands=list(ZE.rglob('BeefCattle_Muzzle_Individualized'))+[ZE]+[p for p in ZE.iterdir() if p.is_dir()]
ZEN_DIR=next((c for c in cands if c.is_dir() and sum(1 for p in c.iterdir() if p.is_dir())>=50), None)
print('hocicos:', ZEN_DIR)

## 5. Modelos + Grad-CAM de similitud (definiciones)

In [ ]:
import numpy as np, torch.nn.functional as F, matplotlib
from PIL import Image
from src.reid.embeddings import EmbeddingExtractor
from src.reid.reid_dataset import entries_from_folders, session_id
try: _JET=matplotlib.colormaps['jet']
except Exception:
    import matplotlib.cm as cm; _JET=cm.get_cmap('jet')
DEVICE='cuda'

class SimGradCAM:
    '''Grad-CAM sobre la similitud coseno de un par (backbone ResNet-50 sin fc).'''
    def __init__(self, backbone):
        self.backbone=backbone
        self.acts=None; self.grads=None
        backbone[7].register_forward_hook(self._f)          # layer4 (última conv)
        backbone[7].register_full_backward_hook(self._b)
    def _f(self,m,i,o): self.acts=o
    def _b(self,m,gi,go): self.grads=go[0]
    def cams(self, x2):                                       # x2: [2,3,H,W]
        self.backbone.zero_grad(set_to_none=True)
        e=F.normalize(self.backbone(x2).flatten(1), dim=1)   # [2,2048]
        sim=(e[0]*e[1]).sum()
        sim.backward()
        a=self.acts.detach(); g=self.grads.detach()
        w=g.mean(dim=(2,3), keepdim=True)
        cam=F.relu((w*a).sum(1))                              # [2,h,w]
        cam=F.interpolate(cam.unsqueeze(1), size=x2.shape[-2:], mode='bilinear',
                          align_corners=False)[:,0].cpu().numpy()
        out=[]
        for i in range(2):
            c=cam[i]-cam[i].min()
            out.append(c/c.max() if c.max()>0 else c)
        return out, float(sim)

# extractores (reusan carga de checkpoint + transforms) → tomamos su backbone
ext_mine=EmbeddingExtractor.from_checkpoint(Path('outputs/checkpoints/cmpd300_source.pt'))
ext_in  =EmbeddingExtractor.from_imagenet()
gc_mine=SimGradCAM(ext_mine.backbone)
gc_in  =SimGradCAM(ext_in.backbone)
TF=ext_mine.tf   # mismo preproc (224 + ImageNet norm) para ambos
SZ=ext_mine.image_size

def overlay(pil, cam):
    base=np.asarray(pil.resize((SZ,SZ)).convert('RGB'))/255.
    return np.clip(0.55*base+0.45*_JET(cam)[...,:3],0,1)

def sample_pairs(root, n, seed=0, distinct_sessions=False):
    '''Pares (img1,img2) del mismo individuo. Si distinct_sessions=True, de SESIONES
    distintas (usa el timestamp del nombre) para evitar comparar fotos gemelas.'''
    ent,_=entries_from_folders(root); by={}
    for e in ent: by.setdefault(e['label'],[]).append(e['path'])
    rng=random.Random(seed); labs=list(by); rng.shuffle(labs)
    ps=[]
    for l in labs:
        v=by[l][:]; rng.shuffle(v)
        if distinct_sessions:
            # elegir dos rutas con session_id distinto
            chosen=None
            for i in range(len(v)):
                for j in range(i+1,len(v)):
                    if session_id(v[i])!=session_id(v[j]):
                        chosen=(v[i],v[j]); break
                if chosen: break
            if chosen is None: continue          # individuo con una sola sesión → se salta
            ps.append((root/chosen[0], root/chosen[1]))
        else:
            if len(v)<2: continue
            ps.append((root/v[0], root/v[1]))
        if len(ps)>=n: break
    return ps
print('listo | image_size',SZ)

## 6. Grad-CAM sobre CARAS (Ahmed)

In [ ]:
import matplotlib.pyplot as plt
def run(root, n_pairs, titulo, distinct_sessions=False):
    pairs=sample_pairs(root, n_pairs, seed=0, distinct_sessions=distinct_sessions)
    for k,(p1,p2) in enumerate(pairs):
        pils=[Image.open(p).convert('RGB') for p in (p1,p2)]
        x=torch.stack([TF(pl) for pl in pils]).to(DEVICE)
        cm_m,s_m=gc_mine.cams(x); cm_i,s_i=gc_in.cams(x)
        fig,ax=plt.subplots(2,3,figsize=(9,6))
        for r in range(2):
            ax[r,0].imshow(pils[r].resize((SZ,SZ))); ax[r,0].set_title('original' if r==0 else '')
            ax[r,1].imshow(overlay(pils[r],cm_m[r])); ax[r,1].set_title(f'MI encoder (sim={s_m:.2f})' if r==0 else '')
            ax[r,2].imshow(overlay(pils[r],cm_i[r])); ax[r,2].set_title(f'ImageNet (sim={s_i:.2f})' if r==0 else '')
            for c in range(3): ax[r,c].axis('off')
        fig.suptitle(f'{titulo} — par {k+1} ({p1.parent.name})', fontsize=10)
        fig.tight_layout(); plt.show()

run(AH, 4, 'CARAS (Ahmed) — sesiones distintas', distinct_sessions=True)

## 7. Grad-CAM sobre HOCICOS (Zenodo)

In [ ]:
run(ZEN_DIR, 4, 'HOCICOS (Zenodo) — al azar (sin timestamp)', distinct_sessions=False)

## Cómo leer esto

Cada fila es una imagen del par; las columnas muestran dónde mira **tu encoder** y dónde mira
**ImageNet** para justificar que las dos son la misma vaca.

- Si en las **caras** los dos miran pelaje / forma de cabeza / fondo (y no el hocico) → explica
  por qué ImageNet empata: la identidad se resuelve por apariencia general, no por biometría de
  hocico.
- Si en los **hocicos** ninguno concentra la atención en la textura fina del hocico → la señal
  biométrica es débil a esa escala/resolución, coherente con que ImageNet iguale.
- Es la evidencia visual que respalda el diagnóstico numérico (ImageNet ≈ tu encoder).